<a href="https://colab.research.google.com/github/aycaaozturk/AML-project/blob/main/AML_clinical_sample_normalize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# AML CLINICAL SAMPLE DATA
# PREPROCESSING FOR SURVIVAL MODELING
#
# One sample per patient
# Binary genomic features retained as 0/1
# FLT3 allelic ratio log-transformed and standardized
# Train-fitted preprocessing applied to validation and test
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# ------------------------------------------------------------
# 1. File paths
# ------------------------------------------------------------

INPUT_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/Clinical Sample/simplified"
)

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Uniklinikum Würzburg/AML/"
    "Output Files 2/Clinical Sample/survival_model_ready"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# Change these filenames if your actual names differ.
TRAIN_PATH = (
    INPUT_DIR / "sample_train/sample_train_one_sample_per_patient.csv"
)

VAL_PATH = (
    INPUT_DIR / "sample_validation/sample_validation_one_sample_per_patient.csv"
)

TEST_PATH = (
    INPUT_DIR / "sample_test/sample_test_one_sample_per_patient.csv"
)


# ------------------------------------------------------------
# 2. Configuration
# ------------------------------------------------------------

ID_COL = "PATIENT_ID"
SAMPLE_ID_COL = "SAMPLE_ID"

FLT3_RATIO_COL = "FLT3_ITD_ALLELIC_RATIO"
FLT3_STATUS_COL = "FLT3_ITD_POSITIVE"

# Recommended main analysis:
INCLUDE_ANALYSIS_COHORT = False
INCLUDE_SAMPLE_SUFFIX = False

# Transform the strongly right-skewed FLT3 ratio.
APPLY_FLT3_RATIO_LOG1P = True

# Remove binary variables that are extremely rare?
# False is safer initially. Feature selection should be performed
# later within the training data or cross-validation.
REMOVE_RARE_BINARY_FEATURES = False

# Minimum number of positive patients required if rare features
# are removed.
MIN_POSITIVE_COUNT = 5


# ------------------------------------------------------------
# 3. Load datasets
# ------------------------------------------------------------

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Original dataset shapes:")
print("Training:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)


Original dataset shapes:
Training: (620, 32)
Validation: (177, 32)
Test: (89, 32)


In [ ]:
# ------------------------------------------------------------
# 4. Basic validation
# ------------------------------------------------------------

required_cols = [
    ID_COL,
    SAMPLE_ID_COL,
    FLT3_STATUS_COL,
    FLT3_RATIO_COL
]

for split_name, dataframe in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    missing_required = [
        col for col in required_cols
        if col not in dataframe.columns
    ]

    if missing_required:
        raise KeyError(
            f"{split_name} is missing required columns: "
            f"{missing_required}"
        )

    if dataframe[ID_COL].isna().any():
        raise ValueError(
            f"{split_name} contains missing patient IDs."
        )

    if dataframe[ID_COL].duplicated().any():
        duplicated_ids = dataframe.loc[
            dataframe[ID_COL].duplicated(keep=False),
            ID_COL
        ].unique()

        raise ValueError(
            f"{split_name} contains multiple rows for some "
            f"patients: {duplicated_ids[:10]}"
        )


# Confirm that patients do not overlap between splits.
train_ids = set(train_df[ID_COL].astype(str))
val_ids = set(val_df[ID_COL].astype(str))
test_ids = set(test_df[ID_COL].astype(str))

assert train_ids.isdisjoint(val_ids), (
    "Patient overlap between training and validation."
)

assert train_ids.isdisjoint(test_ids), (
    "Patient overlap between training and test."
)

assert val_ids.isdisjoint(test_ids), (
    "Patient overlap between validation and test."
)


# ------------------------------------------------------------
# 5. Clean object/string columns
# ------------------------------------------------------------

def clean_text_columns(dataframe):
    """
    Strip unnecessary spaces from text values.
    """
    dataframe = dataframe.copy()

    text_cols = dataframe.select_dtypes(
        include=["object", "string", "category"]
    ).columns

    for col in text_cols:
        dataframe[col] = dataframe[col].apply(
            lambda value:
            value.strip()
            if isinstance(value, str)
            else value
        )

    return dataframe


train_df = clean_text_columns(train_df)
val_df = clean_text_columns(val_df)
test_df = clean_text_columns(test_df)


In [ ]:
# ------------------------------------------------------------
# 6. Check missing values
# ------------------------------------------------------------

print("\nMissing values in training set:")
training_missing = (
    train_df.isna()
    .sum()
    .sort_values(ascending=False)
)

print(training_missing[training_missing > 0])

if training_missing.sum() == 0:
    print("No missing values detected in training set.")



Missing values in training set:
Series([], dtype: int64)
No missing values detected in training set.


In [ ]:
# ------------------------------------------------------------
# 7. Detect constant columns using training data
# ------------------------------------------------------------

constant_cols = [
    col for col in train_df.columns
    if train_df[col].nunique(dropna=False) <= 1
]

print("\nConstant columns detected in training:")
print(constant_cols)

# Expected examples:
# ONCOTREE_CODE
# CANCER_TYPE
# CANCER_TYPE_DETAILED
# SOMATIC_STATUS


Constant columns detected in training:
['ONCOTREE_CODE', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'SOMATIC_STATUS']


In [ ]:
# ------------------------------------------------------------
# 8. Identify binary genomic columns
# ------------------------------------------------------------

NON_GENOMIC_COLUMNS = {
    ID_COL,
    SAMPLE_ID_COL,
    FLT3_RATIO_COL,
    "ONCOTREE_CODE",
    "ANALYSIS_COHORT",
    "CANCER_TYPE",
    "CANCER_TYPE_DETAILED",
    "SOMATIC_STATUS",
    "SAMPLE_SUFFIX"
}

candidate_genomic_cols = [
    col for col in train_df.columns
    if col not in NON_GENOMIC_COLUMNS
]


def is_binary_zero_one(series):
    values = set(
        pd.to_numeric(
            series,
            errors="coerce"
        ).dropna().unique()
    )

    return values.issubset({0, 1})


binary_genomic_cols = [
    col for col in candidate_genomic_cols
    if is_binary_zero_one(train_df[col])
]

nonbinary_candidates = [
    col for col in candidate_genomic_cols
    if col not in binary_genomic_cols
]

print("\nBinary genomic predictors:")
print(binary_genomic_cols)

if nonbinary_candidates:
    print("\nWarning: unexpected non-binary genomic columns:")
    print(nonbinary_candidates)


Binary genomic predictors:
['T_6_9', 'T_8_21', 'T_3_5_Q25_Q34', 'T_6_11_Q27_Q23', 'T_9_11_P22_Q23', 'T_10_11_P11_2_Q23', 'T_11_19_Q23_P13_1', 'INV_16', 'DEL_5Q', 'DEL_7Q', 'DEL_9Q', 'MONOSOMY_5', 'MONOSOMY_7', 'TRISOMY_8', 'TRISOMY_21', 'MLL', 'MINUS_Y', 'MINUS_X', 'FLT3_ITD_POSITIVE', 'FLT3_PM', 'NPM_MUTATION', 'CEBPA_MUTATION', 'WT1_MUTATION']


In [ ]:
# ------------------------------------------------------------
# 9. Validate binary values
# ------------------------------------------------------------

for split_name, dataframe in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    for col in binary_genomic_cols:
        numeric_values = pd.to_numeric(
            dataframe[col],
            errors="coerce"
        )

        invalid_mask = (
            numeric_values.notna()
            & ~numeric_values.isin([0, 1])
        )

        if invalid_mask.any():
            invalid_values = dataframe.loc[
                invalid_mask,
                col
            ].unique()

            raise ValueError(
                f"{split_name}: {col} contains values "
                f"other than 0/1: {invalid_values}"
            )


# ------------------------------------------------------------
# 10. Display binary feature frequencies
# ------------------------------------------------------------

binary_positive_counts = (
    train_df[binary_genomic_cols]
    .apply(pd.to_numeric)
    .sum()
    .sort_values()
)

binary_frequency_table = pd.DataFrame({
    "positive_count": binary_positive_counts,
    "positive_percentage": (
        binary_positive_counts / len(train_df) * 100
    )
})

print("\nBinary genomic feature frequencies:")
print(binary_frequency_table)



Binary genomic feature frequencies:
                   positive_count  positive_percentage
MONOSOMY_5                      1             0.161290
T_3_5_Q25_Q34                   2             0.322581
DEL_5Q                          7             1.129032
T_6_11_Q27_Q23                  9             1.451613
T_6_9                          11             1.774194
DEL_7Q                         11             1.774194
T_10_11_P11_2_Q23              11             1.774194
MONOSOMY_7                     12             1.935484
T_11_19_Q23_P13_1              13             2.096774
TRISOMY_21                     14             2.258065
MINUS_X                        22             3.548387
DEL_9Q                         23             3.709677
MINUS_Y                        28             4.516129
WT1_MUTATION                   41             6.612903
T_9_11_P22_Q23                 42             6.774194
FLT3_PM                        43             6.935484
CEBPA_MUTATION              

In [ ]:

# ------------------------------------------------------------
# 11. Optional removal of very rare binary features
# ------------------------------------------------------------

rare_binary_cols = binary_positive_counts[
    binary_positive_counts < MIN_POSITIVE_COUNT
].index.tolist()

print(
    f"\nBinary variables with fewer than "
    f"{MIN_POSITIVE_COUNT} positive patients:"
)
print(rare_binary_cols)

if REMOVE_RARE_BINARY_FEATURES:
    retained_binary_cols = [
        col for col in binary_genomic_cols
        if col not in rare_binary_cols
    ]
else:
    retained_binary_cols = binary_genomic_cols.copy()

print("\nRetained binary genomic predictors:")
print(retained_binary_cols)



Binary variables with fewer than 5 positive patients:
['MONOSOMY_5', 'T_3_5_Q25_Q34']

Retained binary genomic predictors:
['T_6_9', 'T_8_21', 'T_3_5_Q25_Q34', 'T_6_11_Q27_Q23', 'T_9_11_P22_Q23', 'T_10_11_P11_2_Q23', 'T_11_19_Q23_P13_1', 'INV_16', 'DEL_5Q', 'DEL_7Q', 'DEL_9Q', 'MONOSOMY_5', 'MONOSOMY_7', 'TRISOMY_8', 'TRISOMY_21', 'MLL', 'MINUS_Y', 'MINUS_X', 'FLT3_ITD_POSITIVE', 'FLT3_PM', 'NPM_MUTATION', 'CEBPA_MUTATION', 'WT1_MUTATION']


In [ ]:
# ------------------------------------------------------------
# 12. Validate FLT3 structural relationship
# ------------------------------------------------------------

for split_name, dataframe in [
    ("Training", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    negative_with_positive_ratio = dataframe[
        (dataframe[FLT3_STATUS_COL] == 0)
        & (dataframe[FLT3_RATIO_COL] > 0)
    ]

    if len(negative_with_positive_ratio) > 0:
        print(
            f"\nWarning: {len(negative_with_positive_ratio)} "
            f"{split_name} patients are FLT3-negative but have "
            f"a positive allelic ratio."
        )


In [ ]:
# ------------------------------------------------------------
# 13. Examine FLT3 allelic-ratio skewness
# ------------------------------------------------------------

training_ratio_skewness = (
    train_df[FLT3_RATIO_COL].skew()
)

training_positive_ratio_skewness = train_df.loc[
    train_df[FLT3_STATUS_COL] == 1,
    FLT3_RATIO_COL
].skew()

print("\nFLT3 allelic ratio summary:")
print(train_df[FLT3_RATIO_COL].describe())

print(
    "\nTraining ratio skewness, all patients:",
    training_ratio_skewness
)

print(
    "Training ratio skewness, FLT3-positive patients:",
    training_positive_ratio_skewness
)



FLT3 allelic ratio summary:
count    620.000000
mean       0.122613
std        0.573113
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        9.500000
Name: FLT3_ITD_ALLELIC_RATIO, dtype: float64

Training ratio skewness, all patients: 11.506820099120004
Training ratio skewness, FLT3-positive patients: 5.280530253309579


In [ ]:
# ------------------------------------------------------------
# 14. Create log-transformed FLT3 allelic ratio
# ------------------------------------------------------------

FLT3_LOG_COL = (
    "FLT3_ITD_ALLELIC_RATIO_LOG1P"
)


def add_flt3_log_ratio(dataframe):
    """
    Create log(1 + FLT3 allelic ratio).

    A value of zero remains zero for FLT3-negative patients.
    Positive high ratios are compressed.
    """
    dataframe = dataframe.copy()

    ratio = pd.to_numeric(
        dataframe[FLT3_RATIO_COL],
        errors="coerce"
    )

    if (ratio < 0).any():
        raise ValueError(
            "Negative FLT3 allelic-ratio values were found."
        )

    dataframe[FLT3_LOG_COL] = np.log1p(ratio)

    return dataframe


if APPLY_FLT3_RATIO_LOG1P:
    train_df = add_flt3_log_ratio(train_df)
    val_df = add_flt3_log_ratio(val_df)
    test_df = add_flt3_log_ratio(test_df)

    print(
        "\nSkewness after log1p:",
        train_df[FLT3_LOG_COL].skew()
    )



Skewness after log1p: 4.703509923287689


In [ ]:
# ------------------------------------------------------------
# 15. Define columns excluded from predictors
# ------------------------------------------------------------

DROP_FROM_PREDICTORS = [
    ID_COL,
    SAMPLE_ID_COL,

    # Constant or descriptive metadata:
    "ONCOTREE_CODE",
    "CANCER_TYPE",
    "CANCER_TYPE_DETAILED",
    "SOMATIC_STATUS"
]

# Also remove any other training-set constant columns.
DROP_FROM_PREDICTORS.extend(constant_cols)

if not INCLUDE_ANALYSIS_COHORT:
    DROP_FROM_PREDICTORS.append(
        "ANALYSIS_COHORT"
    )

if not INCLUDE_SAMPLE_SUFFIX:
    DROP_FROM_PREDICTORS.append(
        "SAMPLE_SUFFIX"
    )

# Use transformed FLT3 ratio instead of the raw version.
if APPLY_FLT3_RATIO_LOG1P:
    DROP_FROM_PREDICTORS.append(
        FLT3_RATIO_COL
    )

# Optionally drop rare binary features.
if REMOVE_RARE_BINARY_FEATURES:
    DROP_FROM_PREDICTORS.extend(
        rare_binary_cols
    )

DROP_FROM_PREDICTORS = list(dict.fromkeys(
    col for col in DROP_FROM_PREDICTORS
    if col in train_df.columns
))

print("\nColumns excluded from predictors:")
print(DROP_FROM_PREDICTORS)


Columns excluded from predictors:
['PATIENT_ID', 'SAMPLE_ID', 'ONCOTREE_CODE', 'CANCER_TYPE', 'CANCER_TYPE_DETAILED', 'SOMATIC_STATUS', 'ANALYSIS_COHORT', 'SAMPLE_SUFFIX', 'FLT3_ITD_ALLELIC_RATIO']


In [ ]:
# ------------------------------------------------------------
# 16. Separate identifiers and predictors
# ------------------------------------------------------------

def separate_sample_data(dataframe):
    identifiers = dataframe[
        [ID_COL, SAMPLE_ID_COL]
    ].copy()

    predictors = dataframe.drop(
        columns=DROP_FROM_PREDICTORS
    ).copy()

    return identifiers, predictors


ids_train, X_train = separate_sample_data(
    train_df
)

ids_val, X_val = separate_sample_data(
    val_df
)

ids_test, X_test = separate_sample_data(
    test_df
)


# Ensure all three sets have the same raw predictors.
if X_train.columns.tolist() != X_val.columns.tolist():
    raise ValueError(
        "Training and validation predictor columns differ."
    )

if X_train.columns.tolist() != X_test.columns.tolist():
    raise ValueError(
        "Training and test predictor columns differ."
    )


In [ ]:
# ------------------------------------------------------------
# 17. Explicitly define predictor types
# ------------------------------------------------------------

# Binary features remain 0/1 and are passed through.
binary_cols = [
    col for col in retained_binary_cols
    if col in X_train.columns
]

continuous_cols = []

if APPLY_FLT3_RATIO_LOG1P:
    continuous_cols.append(FLT3_LOG_COL)
else:
    continuous_cols.append(FLT3_RATIO_COL)


# These are optional categorical variables if the user chooses
# to include them.
categorical_cols = []

if (
    INCLUDE_ANALYSIS_COHORT
    and "ANALYSIS_COHORT" in X_train.columns
):
    categorical_cols.append(
        "ANALYSIS_COHORT"
    )

if (
    INCLUDE_SAMPLE_SUFFIX
    and "SAMPLE_SUFFIX" in X_train.columns
):
    # Convert suffix to string so it is treated categorically.
    for dataframe in [X_train, X_val, X_test]:
        dataframe["SAMPLE_SUFFIX"] = (
            dataframe["SAMPLE_SUFFIX"]
            .astype(str)
        )

    categorical_cols.append(
        "SAMPLE_SUFFIX"
    )


# Confirm no predictor was accidentally left unassigned.
assigned_cols = set(
    binary_cols
    + continuous_cols
    + categorical_cols
)

unassigned_cols = [
    col for col in X_train.columns
    if col not in assigned_cols
]

if unassigned_cols:
    raise ValueError(
        "The following predictor columns were not assigned "
        f"to a preprocessing group: {unassigned_cols}"
    )


print("\nBinary columns retained as 0/1:")
print(binary_cols)

print("\nContinuous columns to standardize:")
print(continuous_cols)

print("\nCategorical columns to encode:")
print(categorical_cols)



Binary columns retained as 0/1:
['T_6_9', 'T_8_21', 'T_3_5_Q25_Q34', 'T_6_11_Q27_Q23', 'T_9_11_P22_Q23', 'T_10_11_P11_2_Q23', 'T_11_19_Q23_P13_1', 'INV_16', 'DEL_5Q', 'DEL_7Q', 'DEL_9Q', 'MONOSOMY_5', 'MONOSOMY_7', 'TRISOMY_8', 'TRISOMY_21', 'MLL', 'MINUS_Y', 'MINUS_X', 'FLT3_ITD_POSITIVE', 'FLT3_PM', 'NPM_MUTATION', 'CEBPA_MUTATION', 'WT1_MUTATION']

Continuous columns to standardize:
['FLT3_ITD_ALLELIC_RATIO_LOG1P']

Categorical columns to encode:
[]


In [ ]:
# ------------------------------------------------------------
# 18. Build preprocessing pipelines
# ------------------------------------------------------------

# Binary pipeline:
# - Most-frequent imputation only as a safety fallback
# - No scaling
binary_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        )
    ]
)


# Continuous pipeline:
# - Median fallback
# - Standard scaling using training mean and SD
continuous_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


transformers = [
    (
        "binary",
        binary_pipeline,
        binary_cols
    ),
    (
        "continuous",
        continuous_pipeline,
        continuous_cols
    )
]


# Add a categorical one-hot pipeline only if needed.
if categorical_cols:
    from sklearn.preprocessing import OneHotEncoder

    try:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
            drop=None
        )
    except TypeError:
        encoder = OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
            drop=None
        )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(
                    strategy="constant",
                    fill_value="Missing"
                )
            ),
            (
                "encoder",
                encoder
            )
        ]
    )

    transformers.append(
        (
            "categorical",
            categorical_pipeline,
            categorical_cols
        )
    )


preprocessor = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    verbose_feature_names_out=False
)



In [ ]:
# ------------------------------------------------------------
# 19. Fit only on training data
# ------------------------------------------------------------

X_train_array = preprocessor.fit_transform(
    X_train
)

X_val_array = preprocessor.transform(
    X_val
)

X_test_array = preprocessor.transform(
    X_test
)


# ------------------------------------------------------------
# 20. Restore feature names
# ------------------------------------------------------------

feature_names = (
    preprocessor.get_feature_names_out()
)

X_train_processed = pd.DataFrame(
    X_train_array,
    columns=feature_names,
    index=X_train.index
)

X_val_processed = pd.DataFrame(
    X_val_array,
    columns=feature_names,
    index=X_val.index
)

X_test_processed = pd.DataFrame(
    X_test_array,
    columns=feature_names,
    index=X_test.index
)


# ------------------------------------------------------------
# 21. Restore binary columns as integers
# ------------------------------------------------------------

# ColumnTransformer often returns all values as floats.
# Convert retained binary genomic variables back to 0/1 integers.
for col in binary_cols:
    if col in X_train_processed.columns:
        X_train_processed[col] = (
            X_train_processed[col]
            .round()
            .astype(int)
        )

        X_val_processed[col] = (
            X_val_processed[col]
            .round()
            .astype(int)
        )

        X_test_processed[col] = (
            X_test_processed[col]
            .round()
            .astype(int)
        )


# ------------------------------------------------------------
# 22. Validate processed datasets
# ------------------------------------------------------------

for split_name, predictors in [
    ("Training", X_train_processed),
    ("Validation", X_val_processed),
    ("Test", X_test_processed)
]:
    if predictors.isna().any().any():
        raise ValueError(
            f"Missing values remain in processed "
            f"{split_name} predictors."
        )

    if np.isinf(
        predictors.to_numpy(dtype=float)
    ).any():
        raise ValueError(
            f"Infinite values found in processed "
            f"{split_name} predictors."
        )


assert (
    X_train_processed.columns.tolist()
    == X_val_processed.columns.tolist()
)

assert (
    X_train_processed.columns.tolist()
    == X_test_processed.columns.tolist()
)


print("\nProcessed dataset shapes:")
print("Training:", X_train_processed.shape)
print("Validation:", X_val_processed.shape)
print("Test:", X_test_processed.shape)

print("\nProcessed feature names:")
print(feature_names.tolist())


Processed dataset shapes:
Training: (620, 24)
Validation: (177, 24)
Test: (89, 24)

Processed feature names:
['T_6_9', 'T_8_21', 'T_3_5_Q25_Q34', 'T_6_11_Q27_Q23', 'T_9_11_P22_Q23', 'T_10_11_P11_2_Q23', 'T_11_19_Q23_P13_1', 'INV_16', 'DEL_5Q', 'DEL_7Q', 'DEL_9Q', 'MONOSOMY_5', 'MONOSOMY_7', 'TRISOMY_8', 'TRISOMY_21', 'MLL', 'MINUS_Y', 'MINUS_X', 'FLT3_ITD_POSITIVE', 'FLT3_PM', 'NPM_MUTATION', 'CEBPA_MUTATION', 'WT1_MUTATION', 'FLT3_ITD_ALLELIC_RATIO_LOG1P']


In [ ]:
# ------------------------------------------------------------
# 23. Reattach patient and sample identifiers
# ------------------------------------------------------------

def create_model_ready_sample_file(
    identifiers,
    predictors
):
    identifiers = identifiers.reset_index(
        drop=True
    )

    predictors = predictors.reset_index(
        drop=True
    )

    return pd.concat(
        [identifiers, predictors],
        axis=1
    )


train_model_ready = (
    create_model_ready_sample_file(
        ids_train,
        X_train_processed
    )
)

val_model_ready = (
    create_model_ready_sample_file(
        ids_val,
        X_val_processed
    )
)

test_model_ready = (
    create_model_ready_sample_file(
        ids_test,
        X_test_processed
    )
)


# ------------------------------------------------------------
# 24. Save processed sample datasets
# ------------------------------------------------------------

train_output_path = (
    OUTPUT_DIR
    / "sample_train_survival_model_ready.csv"
)

val_output_path = (
    OUTPUT_DIR
    / "sample_validation_survival_model_ready.csv"
)

test_output_path = (
    OUTPUT_DIR
    / "sample_test_survival_model_ready.csv"
)

train_model_ready.to_csv(
    train_output_path,
    index=False
)

val_model_ready.to_csv(
    val_output_path,
    index=False
)

test_model_ready.to_csv(
    test_output_path,
    index=False
)


# ------------------------------------------------------------
# 25. Save fitted preprocessing object
# ------------------------------------------------------------

preprocessor_path = (
    OUTPUT_DIR
    / "sample_survival_preprocessor.joblib"
)

joblib.dump(
    preprocessor,
    preprocessor_path
)

['/content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/survival_model_ready/sample_survival_preprocessor.joblib']

In [ ]:
# ------------------------------------------------------------
# 26. Save preprocessing metadata
# ------------------------------------------------------------

metadata = {
    "id_column": ID_COL,
    "sample_id_column": SAMPLE_ID_COL,

    "include_analysis_cohort":
        INCLUDE_ANALYSIS_COHORT,

    "include_sample_suffix":
        INCLUDE_SAMPLE_SUFFIX,

    "apply_flt3_ratio_log1p":
        APPLY_FLT3_RATIO_LOG1P,

    "raw_flt3_ratio_skewness":
        float(training_ratio_skewness),

    "positive_only_flt3_ratio_skewness":
        float(training_positive_ratio_skewness),

    "constant_columns_removed":
        constant_cols,

    "columns_excluded_from_predictors":
        DROP_FROM_PREDICTORS,

    "binary_predictors":
        binary_cols,

    "continuous_predictors_scaled":
        continuous_cols,

    "categorical_predictors_encoded":
        categorical_cols,

    "rare_binary_variables":
        rare_binary_cols,

    "remove_rare_binary_features":
        REMOVE_RARE_BINARY_FEATURES,

    "processed_feature_names":
        feature_names.tolist(),

    "n_train":
        len(train_model_ready),

    "n_validation":
        len(val_model_ready),

    "n_test":
        len(test_model_ready)
}


metadata_path = (
    OUTPUT_DIR
    / "sample_survival_preprocessing_metadata.json"
)

with open(
    metadata_path,
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        metadata,
        file,
        indent=2,
        ensure_ascii=False
    )



In [ ]:
# ------------------------------------------------------------
# 27. Final report
# ------------------------------------------------------------

print("\nSample preprocessing completed.")

print("\nSaved files:")
for path in [
    train_output_path,
    val_output_path,
    test_output_path,
    preprocessor_path,
    metadata_path
]:
    print("-", path)


Sample preprocessing completed.

Saved files:
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/survival_model_ready/sample_train_survival_model_ready.csv
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/survival_model_ready/sample_validation_survival_model_ready.csv
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/survival_model_ready/sample_test_survival_model_ready.csv
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/survival_model_ready/sample_survival_preprocessor.joblib
- /content/drive/MyDrive/Uniklinikum Würzburg/AML/Output Files 2/Clinical Sample/survival_model_ready/sample_survival_preprocessing_metadata.json
